# 01. One-box ocean model  
# 1-box 海洋モデルで保存則と時間積分を学ぶ  
# Learning conservation laws and time integration with a one-box ocean model

この Notebook は、2-box モデルへ入る前の橋渡しです。  
This notebook is a bridge to the 2-box model.

海洋をまず **1 つの箱** として扱います。  
We first treat the ocean as **one box**.

## 今日の目標 / Goals

1. 海洋トレーサーを変数として扱う。  
   Treat ocean tracers as variables.

2. 保存則を Python で書く。  
   Write conservation laws in Python.

3. 1 ステップ更新と時間積分を復習する。  
   Review one-step updates and time integration.

4. なぜ 2-box が必要になるのか理解する。  
   Understand why we need a 2-box model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 1-box の概念図 / Conceptual diagram of a one-box model

```text
Atmosphere
   ↑↓ CO2 exchange

Ocean box
  PO4
  DIC
  ALK
  O2
```

このモデルでは、表層・深層・高緯度・低緯度を区別しません。  
This model does not distinguish surface, deep, high-latitude, or low-latitude regions.

これは現実的ではありませんが、プログラムの構造を学ぶには便利です。  
This is not realistic, but it is useful for learning the structure of a program.

In [ ]:
CV1 = 1.0250e3
CV2 = 9.7561e-4
CV3 = 1.0e6
CV4 = 3.1536e7
CV5 = 8.64e4

def to_umolkg(x):
    return CV2 * 1.0e6 * x

## 2. 初期値を入れる / Set initial values

まず、海洋ボックスの中に PO4, DIC, ALK, O2 があるとします。  
First, assume that the ocean box contains PO4, DIC, ALK, and O2.

ここでも、辞書は単なる入れ物です。  
Again, the dictionary is simply a container.

```python
ocean["PO4"]
```

は「海洋ボックスの PO4」を意味します。  
means "PO4 in the ocean box."

In [ ]:
ocean = {
    "PO4": 2.20e-6 * CV1,
    "DIC": 2.258e-3 * CV1,
    "ALK": 2.371e-3 * CV1,
    "O2": 1.70e-4 * CV1,
    "PCO2A": 280.0 / CV3,
}

pd.DataFrame({
    "Quantity": ["PO4", "DIC", "ALK", "O2", "Atmospheric pCO2"],
    "Value": [
        to_umolkg(ocean["PO4"]),
        to_umolkg(ocean["DIC"]),
        to_umolkg(ocean["ALK"]),
        to_umolkg(ocean["O2"]),
        CV3 * ocean["PCO2A"],
    ],
    "Unit": ["umol/kg", "umol/kg", "ueq/kg", "umol/kg", "ppmv"],
})

## 3. ボックスの大きさ / Box size

1-box なので、全海洋を 1 つの箱にします。  
Because this is a one-box model, the entire ocean is one box.

In [ ]:
VOL = 1.292e18
AOCN = 3.49e14

print("Ocean volume =", VOL, "m3")
print("Ocean area   =", AOCN, "m2")

## 4. 生物ポンプを入れる / Add a biological pump

ここでは、単純に生物ポンプで PO4 と DIC が減るとします。  
Here we simply assume that the biological pump decreases PO4 and DIC.

本当は、表層で減った物質は深層で再無機化されます。  
In reality, material removed from the surface is remineralized in the deep ocean.

しかし 1-box では深層がないので、その行き先を表現できません。  
But in a one-box model, there is no deep box, so we cannot represent where it goes.

この不完全さが、次の 2-box モデルへの動機になります。  
This limitation motivates the next 2-box model.

In [ ]:
RCP = 106.0
RO2P = 172.0
CEPS = 0.2  # mol C m-2 yr-1

EP = (CEPS / RCP) * AOCN / CV4  # mol P/s

print("Export production =", EP, "mol P/s")

## 5. 1 日だけ進める / Advance by one day

PO4 は輸出生産によって減るとします。  
Assume that PO4 decreases due to export production.

DIC も炭素として減ります。  
DIC also decreases as carbon is exported.

O2 はここでは簡単化して、有機物分解に対応して減る向きにします。  
For simplicity, O2 is set to decrease as a counterpart of organic matter remineralization.

この 1-box の式は教育用の簡略化です。  
This one-box equation is a teaching simplification.

In [ ]:
DT = 8.64e4  # one day [s]

PO4_new = ocean["PO4"] - EP * DT / VOL
DIC_new = ocean["DIC"] - RCP * EP * DT / VOL
O2_new  = ocean["O2"]  - RO2P * EP * DT / VOL

print("PO4 old/new", to_umolkg(ocean["PO4"]), to_umolkg(PO4_new))
print("DIC old/new", to_umolkg(ocean["DIC"]), to_umolkg(DIC_new))
print("O2  old/new", to_umolkg(ocean["O2"]), to_umolkg(O2_new))

### Mini exercise 1 / ミニ演習 1

`CEPS` を 0.1, 0.2, 0.5 に変えると、1 日後の PO4 変化はどう変わるでしょうか。  
Change `CEPS` to 0.1, 0.2, and 0.5. How does the one-day change in PO4 change?

予想してから実行してください。  
Predict first, then run the cell.

In [ ]:
for ceps in [0.1, 0.2, 0.5]:
    ep = (ceps / RCP) * AOCN / CV4
    po4_new = ocean["PO4"] - ep * DT / VOL
    print(f"CEPS = {ceps:.1f}, PO4 change = {to_umolkg(po4_new - ocean['PO4']):.6f} umol/kg")

## 6. 関数にする / Put it into a function

1 日進める計算を関数にします。  
We put the one-day update into a function.

入力は現在の ocean、出力は 1 日後の ocean です。  
The input is the current ocean, and the output is the ocean after one day.

In [ ]:
def one_step_one_box(ocean, CEPS=0.2):
    RCP = 106.0
    RO2P = 172.0
    EP = (CEPS / RCP) * AOCN / CV4

    new_ocean = dict(ocean)

    new_ocean["PO4"] = ocean["PO4"] - EP * DT / VOL
    new_ocean["DIC"] = ocean["DIC"] - RCP * EP * DT / VOL
    new_ocean["O2"]  = ocean["O2"]  - RO2P * EP * DT / VOL

    return new_ocean

## 7. 時間積分 / Time integration

関数を for 文で繰り返します。  
We repeat the function using a for-loop.

ここでも、for 文は時間積分です。  
Again, the for-loop is time integration.

In [ ]:
def initial_one_box():
    return {
        "PO4": 2.20e-6 * CV1,
        "DIC": 2.258e-3 * CV1,
        "ALK": 2.371e-3 * CV1,
        "O2": 1.70e-4 * CV1,
        "PCO2A": 280.0 / CV3,
    }

def run_one_box(years=500, CEPS=0.2):
    ocean = initial_one_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            history.append({
                "year": day / 365,
                "PO4": to_umolkg(ocean["PO4"]),
                "DIC": to_umolkg(ocean["DIC"]),
                "O2": to_umolkg(ocean["O2"]),
            })

        ocean = one_step_one_box(ocean, CEPS=CEPS)

    return ocean, pd.DataFrame(history)

final, hist = run_one_box()
hist.tail()

## 8. 図で見る / Plot the results

1-box では、PO4, DIC, O2 が一方的に減ります。  
In this one-box model, PO4, DIC, and O2 decrease monotonically.

これは、物質の行き先を表現していないためです。  
This is because the model does not represent where the exported material goes.

In [ ]:
plt.figure()
plt.plot(hist["year"], hist["PO4"])
plt.xlabel("Time [yr]")
plt.ylabel("PO4 [umol/kg]")
plt.title("One-box PO4")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(hist["year"], hist["DIC"])
plt.xlabel("Time [yr]")
plt.ylabel("DIC [umol/kg]")
plt.title("One-box DIC")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(hist["year"], hist["O2"])
plt.xlabel("Time [yr]")
plt.ylabel("O2 [umol/kg]")
plt.title("One-box O2")
plt.grid(True)
plt.show()

## 9. Challenge: 生物ポンプを変える / Change biological export

`CEPS` を変えると、PO4 の減り方が変わります。  
Changing `CEPS` changes the rate of PO4 decrease.

**予想 / Prediction**

`CEPS` を大きくすると、PO4 は速く減るでしょうか、遅く減るでしょうか。  
If `CEPS` is increased, does PO4 decrease faster or slower?

In [ ]:
CEPS_list = [0.05, 0.1, 0.2, 0.5]

plt.figure()
for ceps in CEPS_list:
    _, h = run_one_box(years=300, CEPS=ceps)
    plt.plot(h["year"], h["PO4"], label=f"CEPS={ceps}")

plt.xlabel("Time [yr]")
plt.ylabel("PO4 [umol/kg]")
plt.title("Sensitivity to biological export")
plt.legend()
plt.grid(True)
plt.show()

## 10. なぜ 2-box が必要か / Why do we need a 2-box model?

1-box では、生物ポンプで減った PO4 や DIC の行き先がありません。  
In a one-box model, PO4 and DIC removed by biological export have nowhere to go.

現実には、表層で作られた有機物は沈み、深層で再無機化されます。  
In reality, organic matter produced in the surface ocean sinks and is remineralized in the deep ocean.

だから、次は海洋を次の 2 つに分けます。  
Therefore, next we divide the ocean into two boxes.

```text
Surface ocean
Deep ocean
```

2-box モデルでは、

```text
表層で減る
↓
深層で増える
```

という構造を表現できます。  
In the 2-box model, we can represent:

```text
decrease in surface
↓
increase in deep ocean
```

## 11. 課題 / Exercises

### 課題 1 / Exercise 1

1-box モデルで `CEPS` を大きくすると、PO4, DIC, O2 はどう変化するか。  
What happens to PO4, DIC, and O2 when `CEPS` is increased in the one-box model?

### 課題 2 / Exercise 2

この 1-box モデルの最大の欠点は何か。  
What is the biggest limitation of this one-box model?

### 課題 3 / Exercise 3

なぜ 2-box モデルでは表層と深層を分ける必要があるのか。  
Why do we need to separate surface and deep ocean in the 2-box model?

**想定される答え / Expected answer**

1-box では、表層から取り除かれた物質がどこへ行くのかを表現できません。  
In a one-box model, we cannot represent where material removed from the surface goes.

2-box にすると、表層での生物生産と深層での再無機化を分けて表現できます。  
With a 2-box model, we can separately represent surface biological production and deep remineralization.